In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
df_Limbayat_2018 = pd.read_csv('Surat_AirQuality_Data_Limbayat_2018.csv')
df_Limbayat_2019 = pd.read_csv('Surat_AirQuality_Data_Limbayat_2019.csv')
df_Varachha_2018 = pd.read_csv('Surat_AirQuality_Data_Varachha_2018.csv')
df_Varachha_2019 = pd.read_csv('Surat_AirQuality_Data_Varachha_2019.csv')

print("DataFrame for Limbayat 2018:")
display(df_Limbayat_2018.head())

print("\nDataFrame for Limbayat 2019:")
display(df_Limbayat_2019.head())

print("\nDataFrame for Varachha 2018:")
display(df_Varachha_2018.head())

print("\nDataFrame for Varachha 2019:")
display(df_Varachha_2019.head())

In [ ]:
df_list = [df_Limbayat_2018, df_Limbayat_2019, df_Varachha_2018, df_Varachha_2019]
df_combined = pd.concat(df_list, ignore_index=True)

print("Combined DataFrame:")
display(df_combined.head())
df_combined.info()

In [ ]:
# Drop unnamed columns (index-like columns)
df_combined = df_combined.loc[:, ~df_combined.columns.str.contains('^Unnamed')]

In [ ]:
df_combined

In [ ]:
# Standardize column names (remove spaces, make uppercase, remove special chars)
df_combined.columns = df_combined.columns.str.strip().str.upper().str.replace(" ", "").str.replace(".", "")

print("Available columns after cleaning:")
print(df_combined.columns.tolist())

In [ ]:
# Correct renaming for cleaned uppercase column names
rename_dict = {
    "MONTHLY_MEAN_AVERAGE_CONCENTRATION_PM25": "PM2.5",
    "MONTHLY_MEAN_CONCENTRATION_PM10": "PM10",
    "MONTHLY_MEAN_CONCENTRATION_NO2": "NO2",
    "MONTHLY_MEAN_CONCENTRATION_SO2": "SO2",
    "MONTHLY_MEAN_CONCENTRATION_O3": "O3",
    "NO_OF_AQM_SENSORS_DEPLOYED": "SENSORS"
}

# Apply renaming
df_combined = df_combined.rename(columns=rename_dict)

# Drop non-pollutant columns
df_combined = df_combined.drop(columns=["SENSORS"], errors="ignore")


print("Final dataframe columns after renaming:")
print(df_combined.columns.tolist())

In [ ]:
# -----------------------
# Define CPCB Breakpoints
# -----------------------
breakpoints = {
    "PM2.5": [(0,30,0,50),(31,60,51,100),(61,90,101,200),(91,120,201,300),(121,250,301,400),(251,500,401,500)],
    "PM10":  [(0,50,0,50),(51,100,51,100),(101,250,101,200),(251,350,201,300),(351,430,301,400),(431,500,401,500)],
    "SO2":   [(0,40,0,50),(41,80,51,100),(81,380,101,200),(381,800,201,300),(801,1600,301,400),(1601,2000,401,500)],
    "NO2":   [(0,40,0,50),(41,80,51,100),(81,180,101,200),(181,280,201,300),(281,400,301,400),(401,500,401,500)],
    "O3":    [(0,50,0,50),(51,100,51,100),(101,168,101,200),(169,208,201,300),(209,748,301,400),(749,1000,401,500)]
}

# Function to compute sub-index
def get_sub_index(pollutant, value):
    if pd.isna(value):
        return None
    for (B_LO, B_HI, I_LO, I_HI) in breakpoints[pollutant]:
        if B_LO <= value <= B_HI:
            return ((I_HI - I_LO) / (B_HI - B_LO)) * (value - B_LO) + I_LO
    return None

In [ ]:
# Compute Sub-Indices for each pollutant
for pollutant in breakpoints.keys():
    df_combined[f"{pollutant}_SubIndex"] = df_combined[pollutant].apply(
        lambda x: get_sub_index(pollutant, x) if pd.notnull(x) else None
    )

# Now compute AQI as the max of all sub-indices
def calculate_aqi(row):
    sub_indices = [
        row[f"{pollutant}_SubIndex"]
        for pollutant in breakpoints.keys()
        if pd.notnull(row[f"{pollutant}_SubIndex"])
    ]
    return max(sub_indices) if sub_indices else None


In [ ]:
df_combined["AQI"] = df_combined.apply(calculate_aqi, axis=1)

In [ ]:
# Preview
print(df_combined[[col for col in df_combined.columns if "SubIndex" in col or col=="AQI"]].head())

In [ ]:
df_combined

In [ ]:
# AQI Categories
def aqi_category(aqi):
    if pd.isna(aqi): return None
    if aqi <= 50: return "Good"
    elif aqi <= 100: return "Satisfactory"
    elif aqi <= 200: return "Moderate"
    elif aqi <= 300: return "Poor"
    elif aqi <= 400: return "Very Poor"
    else: return "Severe"

df_combined["AQI_Category"] = df_combined["AQI"].apply(aqi_category)


In [ ]:
df_combined

In [ ]:

# -----------------------
# Define target
# -----------------------
y = df_combined["AQI_Category"]

In [ ]:
# Drop target column only (keep AQI + pollutants as features)
X = df_combined.drop(columns=["AQI", "AQI_Category", "PM2.5_SubIndex",	"PM10_SubIndex",	"SO2_SubIndex",	"NO2_SubIndex",	"O3_SubIndex"])

In [ ]:
X

In [ ]:
# Encode categorical columns (like City, Location) into numeric dummies
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
if categorical_cols:
    X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

In [ ]:
X

In [ ]:

# Encode target
le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [ ]:
# Ensure each class has at least 2 samples
class_counts = np.bincount(y_encoded)

In [ ]:
class_counts

In [ ]:
valid_idx = np.isin(y_encoded, np.where(class_counts >= 2)[0])
X = X.iloc[valid_idx]
y_encoded = y_encoded[valid_idx]


In [ ]:
y_encoded

In [ ]:


# # Train-test split
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded
# )

# # Standardize features
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)

# # -----------------------
# # Define models
# # -----------------------
# models = {
#     "Decision Tree": (DecisionTreeClassifier(random_state=42), X_train, X_test),
#     "Logistic Regression": (LogisticRegression(multi_class="multinomial", max_iter=1000, random_state=42),
#                              X_train_scaled, X_test_scaled),
#     "Naive Bayes": (GaussianNB(), X_train_scaled, X_test_scaled)
# }

# results = {}

# # -----------------------
# # Training & Evaluation
# # -----------------------
# for name, (model, Xtr, Xte) in models.items():
#     model.fit(Xtr, y_train)
#     y_pred = model.predict(Xte)

#     acc = accuracy_score(y_test, y_pred)
#     results[name] = acc

#     print(f"\n🔹 {name} Performance:")
#     print(classification_report(
#         y_test, y_pred,
#         labels=np.arange(len(le.classes_)),
#         target_names=le.classes_
#     ))

#     # Confusion Matrix
#     cm = confusion_matrix(y_test, y_pred, labels=np.arange(len(le.classes_)))
#     sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
#                 xticklabels=le.classes_, yticklabels=le.classes_)
#     plt.title(f"Confusion Matrix - {name}")
#     plt.xlabel("Predicted")
#     plt.ylabel("True")
#     plt.show()

#     # Misclassification Analysis
#     misclassified = y_test != y_pred
#     plt.bar(["Correct", "Incorrect"], [sum(~misclassified), sum(misclassified)],
#             color=["green","red"])
#     plt.title(f"Classification Accuracy Breakdown - {name}")
#     plt.ylabel("Count")
#     plt.show()

# # -----------------------
# # Compare Accuracies
# # -----------------------
# plt.bar(results.keys(), results.values(), color="skyblue")
# plt.title("Model Accuracy Comparison")
# plt.ylabel("Accuracy")
# plt.show()


In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded
)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# -----------------------
# Hyperparameter tuning for Decision Tree
# -----------------------
param_grid = {
    "criterion": ["gini", "entropy", "log_loss"],
    "max_depth": [3, 5, 7, 10, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "random_state": [42]
}

dt = DecisionTreeClassifier()
grid_search = GridSearchCV(dt, param_grid, cv=5, scoring="accuracy", n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)

best_dt = grid_search.best_estimator_
print("Best Decision Tree Parameters:", grid_search.best_params_)
print("Best Cross-Validation Accuracy:", grid_search.best_score_)

# -----------------------
# Define models with tuned Decision Tree
# -----------------------
models = {
    "Decision Tree (Tuned)": (best_dt, X_train_scaled, X_test_scaled),
    "Logistic Regression": (LogisticRegression(multi_class="multinomial", max_iter=1000, random_state=42),
                             X_train_scaled, X_test_scaled),
    "Naive Bayes": (GaussianNB(), X_train_scaled, X_test_scaled)
}

results = {}

# -----------------------
# Training & Evaluation
# -----------------------
for name, (model, Xtr, Xte) in models.items():
    # Cross-validation
    scores = cross_val_score(model, Xtr, y_train, cv=5, scoring="accuracy")
    print(f"\n{name} - CV Mean Accuracy: {scores.mean():.4f} (+/- {scores.std():.4f})")

    # Fit and predict
    model.fit(Xtr, y_train)
    y_pred = model.predict(Xte)

    acc = accuracy_score(y_test, y_pred)
    results[name] = acc

    print(f"\n🔹 {name} Performance:")
    print(classification_report(
        y_test, y_pred,
        labels=np.arange(len(le.classes_)),
        target_names=le.classes_
    ))

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred, labels=np.arange(len(le.classes_)))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=le.classes_, yticklabels=le.classes_)
    plt.title(f"Confusion Matrix - {name}")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.show()

    # Misclassification Analysis
    misclassified = y_test != y_pred
    plt.bar(["Correct", "Incorrect"], [sum(~misclassified), sum(misclassified)],
            color=["green","red"])
    plt.title(f"Classification Accuracy Breakdown - {name}")
    plt.ylabel("Count")
    plt.show()

# -----------------------
# Compare Accuracies
# -----------------------
plt.bar(results.keys(), results.values(), color="skyblue")
plt.title("Model Accuracy Comparison")
plt.ylabel("Accuracy")
plt.show()

In [ ]:
import joblib

# Save the best Decision Tree model
joblib.dump(best_dt, "best_decision_tree.pkl")

# Save the scaler (used for standardization)
joblib.dump(scaler, "scaler.pkl")

# Save the label encoder (to convert back category names)
joblib.dump(le, "label_encoder.pkl")

print("✅ Model, Scaler, and Label Encoder saved successfully!")
